# Semana 9 — Projeto Prático de Data Warehouse e Modelagem Dimensional - SQL

Mini Data Warehouse de e-commerce usando o dataset Olist.

## Objetivos

- Diferenciar OLTP e OLAP
- Executar ETL com Pandas
- Criar dimensões e tabela fato
- Identificar PK, FK e Surrogate Key
- Criar Star Schema
- Consultar o modelo com SQL
- Interpretar métricas de BI

## Arquivos necessários

Baixe no Kaggle o dataset Brazilian E-Commerce Public Dataset by Olist e coloque na pasta do notebook:

- olist_orders_dataset.csv
- olist_order_items_dataset.csv
- olist_customers_dataset.csv
- olist_products_dataset.csv
- product_category_name_translation.csv

In [1]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

# 1. Cenário de negócio

O diretor quer responder:

1. Qual foi o faturamento total?
2. Quais estados mais faturaram?
3. Quais categorias mais venderam?
4. Qual foi o ticket médio?
5. Como o faturamento evoluiu ao longo do tempo?

# 2. Extract — carregando fontes OLTP

In [2]:
# Voltando um nível para sair da pasta 'Exercício' e entrar na pasta 'Dataset'
pedidos = pd.read_csv('../Dataset/olist_orders_dataset.csv')
itens = pd.read_csv('../Dataset/olist_order_items_dataset.csv')
clientes = pd.read_csv('../Dataset/olist_customers_dataset.csv')
produtos = pd.read_csv('../Dataset/olist_products_dataset.csv')
traducao_categoria = pd.read_csv('../Dataset/product_category_name_translation.csv')

# Validando o carregamento
print('orders:', pedidos.shape)
print('items:', itens.shape)
print('customers:', clientes.shape)
print('products:', produtos.shape)
print('category_translation:', traducao_categoria.shape)

orders: (99441, 8)
items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)
category_translation: (71, 2)


## Exercício 1 🟢 Fácil — OLTP

Responda:

1. Por que essas tabelas podem ser consideradas fontes OLTP?
    - Porque estão vindo direto do sistema operacional e transacional do e-commerce. Foram desenhadas para registrar dados em tempo real (pedidos, itens no carrinho..), priorizando velocidade de venda e não a análise dos dados. 
2. Qual tabela registra o pedido? 
    - pedidos
3. Qual tabela registra os itens de cada pedido?
    - itens
4. Qual tabela descreve os clientes?
    - clientes
5. Qual tabela descreve os produtos?
    - produtos

# 3. Análise exploratória inicial

In [ ]:
pedidos.head()

In [ ]:
itens.head()

In [ ]:
clientes.head()

In [ ]:
produtos.head()

## Exercício 2 🟢 Fácil — Explorando os dados

Use `.info()`, `.shape` e `.isnull().sum()` para investigar as tabelas.

In [ ]:
# Investigue a tabela orders
# infos - informações gerais sobre a tabela
print('\nTabela de Pedidos:')
pedidos.info()

# shape - tamanho da tabela
print('\nShape:')
print(pedidos.shape)

# nulls - valores nulos
print('\nValores nulos:')
pedidos.isnull().sum()

In [ ]:
# Investigue a tabela items

# infos - informações gerais sobre a tabela
print('\nTabela de Itens:')
itens.info()

# shape - tamanho da tabela
print('\nShape:')
print(itens.shape)

# nulls - valores nulos
print('\nValores nulos:')
itens.isnull().sum()

In [ ]:
# Investigue a tabela customers
# SUA RESPOSTA AQUI

# infos - informações gerais sobre a tabela
print('\nTabela de Clientes:')
clientes.info()

# shape - tamanho da tabela
print('\nShape:')
print(clientes.shape)

# nulls - valores nulos
print('\nValores nulos:')
clientes.isnull().sum()

In [ ]:
# Investigue a tabela products
# SUA RESPOSTA AQUI

# infos - informações gerais sobre a tabela
print('\nTabela de Produtos:')
produtos.info()

# shape - tamanho da tabela
print('\nShape:')
print(produtos.shape)

# nulls - valores nulos
print('\nValores nulos:')
produtos.isnull().sum()

# 4. Transform — tratamento dos dados

Transformações comuns: converter tipos, remover duplicidades, tratar nulos, padronizar nomes e criar colunas derivadas.

In [ ]:
pedidos['order_purchase_timestamp'] = pd.to_datetime(pedidos['order_purchase_timestamp'])
pedidos['order_approved_at'] = pd.to_datetime(pedidos['order_approved_at'])
pedidos['order_delivered_customer_date'] = pd.to_datetime(pedidos['order_delivered_customer_date'])
pedidos['order_estimated_delivery_date'] = pd.to_datetime(pedidos['order_estimated_delivery_date'])

pedidos[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date']].head()

## Exercício 3 🟢 Fácil — Criando atributos de tempo

Crie as colunas `ano`, `mes`, `dia`, `trimestre` e `ano_mes` a partir de `order_purchase_timestamp`.

In [ ]:
# SUA RESPOSTA AQUI

# Conversão dos textos para datas (Datetime)
pedidos['order_purchase_timestamp'] = pd.to_datetime(pedidos['order_purchase_timestamp'])
pedidos['order_approved_at'] = pd.to_datetime(pedidos['order_approved_at'])
pedidos['order_delivered_customer_date'] = pd.to_datetime(pedidos['order_delivered_customer_date'])
pedidos['order_estimated_delivery_date'] = pd.to_datetime(pedidos['order_estimated_delivery_date'])

# Extraindo ano, mês, dia e trimestre da data de compra
pedidos['ano'] = pedidos['order_purchase_timestamp'].dt.year
pedidos['mes'] = pedidos['order_purchase_timestamp'].dt.month
pedidos['dia'] = pedidos['order_purchase_timestamp'].dt.day
pedidos['trimestre'] = pedidos['order_purchase_timestamp'].dt.quarter

# Cria formato Ano-Mês
pedidos['ano_mes'] = pedidos['order_purchase_timestamp'].dt.to_period('M').astype(str)

#Visualizando o resultado das novas colunas
pedidos[['order_purchase_timestamp', 'ano', 'mes', 'dia', 'trimestre', 'ano_mes']].head()

## Exercício 4 🟢 Fácil — Remoção de duplicidades

Verifique se existem duplicidades nas tabelas principais e remova-as.

In [ ]:
# SUA RESPOSTA AQUI

# Verificar a quantidade de linhas duplicadas antes de remover
print('Linhas duplicadas em pedidos antes de remover:', pedidos.duplicated().sum())
print('Linhas duplicadas em itens antes de remover:', itens.duplicated().sum())
print('Linhas duplicadas em clientes antes de remover:', clientes.duplicated().sum())
print('Linhas duplicadas em produtos antes de remover:', produtos.duplicated().sum())

# Remover linhas duplicadas caso existam para garantir integridade dos dados
pedidos = pedidos.drop_duplicates()
itens = itens.drop_duplicates()
clientes = clientes.drop_duplicates()
produtos = produtos.drop_duplicates()

print('\n ========================================= \n')

# Verificar a quantidade de linhas duplicadas depois de remover
print('Linhas duplicadas em pedidos depois de remover:', pedidos.duplicated().sum())
print('Linhas duplicadas em itens depois de remover:', itens.duplicated().sum())
print('Linhas duplicadas em clientes depois de remover:', clientes.duplicated().sum())
print('Linhas duplicadas em produtos depois de remover:', produtos.duplicated().sum())


## Exercício 5 🟡 Médio — Filtro de pedidos válidos

Considere apenas pedidos com status `delivered` e crie `produtos_entregue`.

In [ ]:
# SUA RESPOSTA AQUI

# Filtrar pedidos com status 'delivered'
pedidos_validados = pedidos[pedidos['order_status'] == 'delivered'].copy()

# Validar se o filtro foi aplicado corretamente exibindo a quantidade de linhas 
print("Quantidade de pedidos entregues:", pedidos_validados.shape)


# 5. Modelagem dimensional

Modelo esperado:

```text
dim_cliente
     |
dim_produto -- fato_vendas -- dim_tempo
```

A tabela fato guarda eventos mensuráveis. As dimensões guardam contexto.

# 6. Criando a dimensão cliente

In [ ]:
dim_cliente = clientes[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']].drop_duplicates().copy()
dim_cliente = dim_cliente.reset_index(drop=True)
dim_cliente['cliente_sk'] = dim_cliente.index + 1
dim_cliente = dim_cliente[['cliente_sk', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state']]
dim_cliente.head()

## Exercício 6 🟡 Médio — PK, SK e chave natural

1. Qual coluna é a Surrogate Key da dimensão cliente?
    - cliente_sk
2. Qual coluna veio do sistema de origem?
    - customer_id e customer_unique_id
3. Por que não usar CPF, e-mail ou código da origem como PK física do DW?
    - O CPF não deve ser usado como chave primária física no Data Warehouse porque chaves em formato de texto tornam as buscas e os cruzamentos de tabelas muito mais lentos se comparados a números inteiros. Além disso, se o cliente alterar o cadastro ou se o sistema de origem mudar as regras de validação, o modelo analítico perderá o histórico de compras do passado. O uso da Surrogate Key numérico-sequencial garante a velocidade das consultas e o isolamento completo contra alterações nos sistemas de origem.

# 7. Criando a dimensão produto

In [ ]:
dim_produto = produtos[['product_id', 'product_category_name']].drop_duplicates().copy()
dim_produto = dim_produto.merge(traducao_categoria, on='product_category_name', how='left')
dim_produto['product_category_name_english'] = dim_produto['product_category_name_english'].fillna('unknown')
dim_produto = dim_produto.reset_index(drop=True)
dim_produto['produto_sk'] = dim_produto.index + 1
dim_produto = dim_produto[['produto_sk', 'product_id', 'product_category_name', 'product_category_name_english']]
dim_produto.head()

## Exercício 7 🟡 Médio — Star Schema vs Snowflake

1. Esse modelo se parece mais com Star Schema ou Snowflake?
    - Com o Star Schema. A tabela de produtos veio com o nome da categoria acoplado a ela. Não foi necessário pular de tabela e tabela para buscar a categoria do produto, está tudo junto em um lugar só. 
2. Como ficaria se categoria fosse uma tabela separada?
    - Snowflake. Seria um efeito dominó, para descobrir a categoria de uma venda, o sistema teria que olhar primeiro a tabela de vendas, depois pular para a tabela de produtos e só então dar mais um pulo para a tabela de categoria. Fica tudo ramificado e espalhado.
3. Qual opção costuma ser mais simples para Power BI?
    O Star Schema é mais simples. Como as informações estão mais unidas , o Power BI não precisa fazer malabarismo cruzando várias tabelas separadas. Isso deixa os gráficos mais rápidos de carregar e fáceis de criar relatórios.

# 8. Criando a dimensão tempo

In [ ]:
#verifica se tem duplicidade de data e copia a tabela para criar a dimensão tempo

dim_tempo = pedidos_validados[['order_purchase_timestamp']].drop_duplicates().copy()

### Crie as colunas necessárias para a dimensão tempo
dim_tempo['data'] = dim_tempo['order_purchase_timestamp'].dt.date #cria a coluna data apenas com a data, sem o horário
dim_tempo['ano'] = dim_tempo['order_purchase_timestamp'].dt.year #cria a coluna ano apenas com o ano da data
dim_tempo['mes'] = dim_tempo['order_purchase_timestamp'].dt.month #cria a coluna mes apenas com o mês da data
dim_tempo['dia'] = dim_tempo['order_purchase_timestamp'].dt.day #cria a coluna dia apenas com o dia da data
dim_tempo['trimestre'] = dim_tempo['order_purchase_timestamp'].dt.quarter #cria a coluna trimestre apenas com o trimestre da data


dim_tempo['ano_mes'] = dim_tempo['order_purchase_timestamp'].dt.to_period('M').astype(str) #cria a coluna ano_mes apenas com o ano e o mês da data, no formato "YYYY-MM"
dim_tempo = dim_tempo.drop_duplicates('data').reset_index(drop=True) #remove as linhas duplicadas com base na coluna data e reseta o índice
dim_tempo['tempo_sk'] = dim_tempo.index + 1 #cria a coluna tempo_sk com os valores sequenciais
dim_tempo = dim_tempo[['tempo_sk', 'data', 'ano', 'mes', 'dia', 'trimestre', 'ano_mes']] #reordena as colunas da dimensão tempo
dim_tempo.head() #exibe as primeiras linhas da dimensão tempo

## Exercício 8 🟡 Médio — Dimensão tempo

1. Por que uma dimensão tempo é útil em BI?
    - Porque ela permite analisar os dados ao longo do tempo de forma organizada.
2. Que análises ela permite?
    - Permite analisar a evolução do faturamento, comparar períodos, identificar sazonalidade, calcular vendas por mês, trimestre ou ano e acompanhar tendências de crescimento.
3. Qual coluna é a SK da dimensão tempo?
    - A Surrogate Key da dimensão tempo é a coluna tempo_sk.

# 9. Criando a tabela fato

Evento: um item vendido dentro de um pedido.

Granularidade: uma linha por item vendido em um pedido.

In [ ]:
fato_vendas = itens[['order_id', 'order_item_id', 'product_id', 'price', 'freight_value']].copy()
fato_vendas = fato_vendas.merge(pedidos_validados[['order_id', 'customer_id', 'order_purchase_timestamp']], on='order_id', how='inner')
fato_vendas.head()

## Exercício 9 🟡 Médio — Relacionando fato e dimensões

Complete a construção da `fato_vendas` trazendo `cliente_sk`, `produto_sk` e `tempo_sk`.

In [ ]:
fato_vendas = fato_vendas.merge(dim_cliente[['cliente_sk', 'customer_id']], on='customer_id', how='left')
fato_vendas = fato_vendas.merge(dim_produto[['produto_sk', 'product_id']], on='product_id', how='left')
fato_vendas['data'] = fato_vendas['order_purchase_timestamp'].dt.date
fato_vendas = fato_vendas.merge(dim_tempo[['tempo_sk', 'data']], on='data', how='left')
fato_vendas.head()

## Exercício 10 🟡 Médio — Métricas da fato

Crie `valor_produto`, `valor_frete` e `valor_total`.

In [ ]:
fato_vendas['valor_produto'] = fato_vendas['price']
fato_vendas['valor_frete'] = fato_vendas['freight_value']
fato_vendas['valor_total'] = fato_vendas['valor_produto'] + fato_vendas['valor_frete']

## Exercício 11 🔴 Difícil — Fato final

Deixe a fato com: `order_id`, `order_item_id`, `cliente_sk`, `produto_sk`, `tempo_sk`, `valor_produto`, `valor_frete`, `valor_total`.

Depois, crie `venda_sk`.

In [ ]:
fato_vendas_final = fato_vendas[['order_id','order_item_id','cliente_sk','produto_sk','tempo_sk','valor_produto','valor_frete','valor_total']].copy()
fato_vendas_final = fato_vendas_final.reset_index(drop=True)
fato_vendas_final['venda_sk'] = fato_vendas_final.index + 1
fato_vendas_final = fato_vendas_final[['venda_sk','order_id','order_item_id','cliente_sk','produto_sk','tempo_sk','valor_produto','valor_frete','valor_total']]
fato_vendas_final.head()

# 10. Validando o Star Schema

## Exercício 12 🔴 Difícil — Documentação do modelo


| Tabela | Tipo | PK | FK | Descrição |
|---|---|---|---|---|
| dim_cliente | Dimensão | cliente_sk | - | Contexto do cliente |
| dim_produto | Dimensão | produto_sk | - | Contexto do produto |
| dim_tempo | Dimensão | tempo_sk | - | Contexto temporal |
| fato_vendas | Fato | venda_sk | cliente_sk, produto_sk, tempo_sk | Evento mensurável de venda |

Granularidade: uma linha por item vendido por pedido.

# 11. SQL Analítico com DuckDB

In [ ]:
con = duckdb.connect()
con.register('dim_cliente', dim_cliente) # registra a dimensão cliente no banco de dados
con.register('dim_produto', dim_produto) # registra a dimensão produto no banco de dados
con.register('dim_tempo', dim_tempo) # registra a dimensão tempo no banco de dados
con.register('fato_vendas', fato_vendas_final) # registra a tabela fato vendas no banco de dados

# Semana 9 — SQL para Data Warehouse

Nesta semana, vamos sair da etapa de construção do modelo dimensional e usar o Star Schema para responder perguntas de BI com SQL.

## Objetivos da Semana 9

Ao final das 3 aulas, você deverá conseguir:

- Usar `SELECT`, `WHERE` e `ORDER BY` em consultas analíticas.
- Fazer `INNER JOIN` entre tabela fato e dimensões.
- Usar `LEFT JOIN` para investigar ausência de relacionamento.
- Criar métricas com `GROUP BY`, `SUM`, `COUNT` e `AVG`.
- Filtrar agregações com `HAVING`.
- Usar funções de data como `DATE_TRUNC` e `EXTRACT`.

## Aula 1 da Semana 9 — SQL básico em DW

### Roteiro teórico

Em um Data Warehouse, o SQL é usado principalmente para análise.  
Diferente de um sistema transacional, onde o foco é inserir e atualizar registros, aqui o foco é consultar, agregar e transformar dados em indicadores.

A estrutura básica de uma consulta é:

```sql
SELECT colunas
FROM tabela
WHERE condição
ORDER BY coluna;
```

No nosso modelo:

- `fato_vendas` guarda os eventos mensuráveis de venda.
- `dim_cliente` explica quem comprou e onde.
- `dim_produto` explica o que foi vendido.
- `dim_tempo` permite analisar a venda no tempo.

## Consulta exemplo — Faturamento total

In [ ]:
con.sql("""
SELECT SUM(valor_total) AS faturamento_total
FROM fato_vendas 
""").df() # consulta para calcular o faturamento total da loja, somando a coluna valor_total da tabela fato_vendas

### Exemplo slide 5 - semana 9

In [ ]:
con.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas
LIMIT 20
""").df() # consulta para exibir as 20 primeiras linhas da tabela fato_vendas, mostrando as colunas venda_sk, order_id, valor_produto, valor_frete e valor_total

### Exemplo slide 7 semana 9

In [ ]:
con.sql("""SELECT
   venda_sk,
   order_id,
   valor_produto,
   valor_frete,
   valor_total
FROM fato_vendas
WHERE valor_total>500
ORDER BY valor_total DESC
LIMIT 20
""").df() # consulta para exibir as 20 vendas mais caras da tabela fato_vendas, mostrando as colunas venda_sk, order_id, valor_produto, valor_frete e valor_total, filtrando apenas as vendas com valor_total maior que 500 e ordenando pelo valor_total em ordem decrescente

### Exemplo — usando `WHERE` e `ORDER BY`

A consulta abaixo busca vendas com valor total maior que 200 e ordena da maior para a menor.

In [ ]:
con.sql("""
SELECT
    venda_sk,
    order_id,
    valor_total
FROM fato_vendas
WHERE valor_total > 200
ORDER BY valor_total DESC
LIMIT 20
""").df() # consulta para exibir as 20 vendas mais caras da tabela fato_vendas, mostrando as colunas venda_sk, order_id e valor_total, filtrando apenas as vendas com valor_total maior que 200 e ordenando pelo valor_total em ordem decrescente

## Exercício 13 🟢 Fácil — SELECT em tabela fato

Liste as colunas `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total` das 15 primeiras vendas.

In [ ]:
con.sql("""
    SELECT 
        venda_sk, -- seleciona a coluna venda_sk da tabela fato_vendas
        order_id, -- seleciona a coluna order_id da tabela fato_vendas
        valor_produto, -- seleciona a coluna valor_produto da tabela fato_vendas
        valor_frete, -- seleciona a coluna valor_frete da tabela fato_vendas
        valor_total -- seleciona a coluna valor_total da tabela fato_vendas
    FROM fato_vendas -- seleciona a tabela fato_vendas para realizar a consulta
    LIMIT 15
""").df() # consulta para exibir as 15 primeiras linhas da tabela fato_vendas, mostrando as colunas venda_sk, order_id, valor_produto, valor_frete e valor_total

## Exercício 14 🟢 Fácil — Filtro com WHERE

Liste as vendas cujo `valor_frete` seja maior que 50.  
Mostre `venda_sk`, `order_id`, `valor_frete` e `valor_total`.

In [ ]:
con.sql("""
    SELECT
        venda_sk, -- seleciona a coluna venda_sk da tabela fato_vendas
        order_id, -- seleciona a coluna order_id da tabela fato_vendas
        valor_produto, -- seleciona a coluna valor_produto da tabela fato_vendas
        valor_frete, -- seleciona a coluna valor_frete da tabela fato_vendas
        valor_total -- seleciona a coluna valor_total da tabela fato_vendas
    FROM fato_vendas -- seleciona a tabela fato_vendas para realizar a consulta
    WHERE valor_total > 50 -- filtra as linhas onde o valor_total é maior que 50
""").df()

## 🟢 Desafio Fácil — Filtro e Ordenação

Selecionar vendas com `valor_total` acima de um limite (vamos usar acima de 100 como exemplo padrão) e ordenar do maior para o menor.

In [ ]:
con.sql("""
    SELECT
        venda_sk, -- seleciona a coluna venda_sk da tabela fato_vendas
        order_id, -- seleciona a coluna order_id da tabela fato_vendas
        valor_total -- seleciona a coluna valor_total da tabela fato_vendas
    FROM fato_vendas -- seleciona a tabela fato_vendas como fonte dos dados
    WHERE valor_total > 100 -- filtra as linhas onde o valor_total é maior que 100
    LIMIT 20
""").df()

## 🟡 Desafio Médio — Vendas por Categoria Específica

Consultar vendas de uma categoria específica unindo `fato_vendas` e `dim_produto` (vamos filtrar a categoria `'health_beauty'` como exemplo).

In [ ]:
con.sql("""
    SELECT 
        f.venda_sk, -- seleciona a coluna venda_sk da tabela fato_vendas, referenciada como f
        f.order_id, -- seleciona a coluna order_id da tabela fato_vendas, referenciada como f
        p.product_category_name_english AS categoria, -- seleciona a coluna product_category_name_english da tabela dim_produto, referenciada como p, e renomeia como categoria
        f.valor_total -- seleciona a coluna valor_total da tabela fato_vendas, referenciada como f
    FROM fato_vendas f -- seleciona a tabela fato_vendas e a referencia como f para facilitar a escrita da consulta
    INNER JOIN dim_produto p -- realiza um inner join com a tabela dim_produto, referenciada como p, para combinar os dados das duas tabelas com base na chave produto_sk
        ON f.produto_sk = p.produto_sk -- especifica a condição de junção, onde a coluna produto_sk da tabela fato_vendas deve ser igual à coluna produto_sk da tabela dim_produto
    WHERE p.product_category_name_english = 'health_beauty' -- filtra as linhas onde a categoria do produto é igual a 'health_beauty'
""").df()

## 🔴 Desafio Difícil — Clientes Sem Vendas (LEFT JOIN)
Enunciado: Identificar clientes sem vendas usando LEFT JOIN.

- O LEFT JOIN traz todos os clientes da tabela de clientes, mesmo os que nunca compraram nada. 
Quando filtramos onde a `f.venda_sk` IS NULL, o banco encontra exatamente quem está com a folha de compras em branco!

In [ ]:
con.sql("""
    SELECT 
        c.cliente_sk, -- seleciona a coluna cliente_sk da tabela dim_cliente, referenciada como c
        c.customer_id, -- seleciona a coluna customer_id da tabela dim_cliente, referenciada como c
        f.venda_sk -- seleciona a coluna venda_sk da tabela fato_vendas, referenciada como f
    FROM dim_cliente c -- seleciona a tabela dim_cliente e a referencia como c para facilitar a escrita da consulta
    LEFT JOIN fato_vendas f  -- realiza um left join com a tabela fato_vendas, referenciada como f, para combinar os dados das duas tabelas com base na chave cliente_sk
        ON c.cliente_sk = f.cliente_sk -- especifica a condição de junção, onde a coluna cliente_sk da tabela dim_cliente deve ser igual à coluna cliente_sk da tabela fato_vendas
    WHERE f.venda_sk IS NULL
""").df()

## 🏆 Desafio BI — Top 10 Produtos por Receita
Montar uma consulta que responda "top 10 produtos por receita (faturamento)".

In [ ]:
con.sql("""
    SELECT 
        f.produto_sk, -- seleciona a coluna produto_sk da tabela fato_vendas, referenciada como f
        p.product_category_name_english AS categoria, -- seleciona a coluna product_category_name_english da tabela dim_produto, referenciada como p, e renomeia como categoria
        f.valor_total AS receita -- seleciona a coluna valor_total da tabela fato_vendas, referenciada como f, e renomeia como receita
    FROM fato_vendas f -- seleciona a tabela fato_vendas e a referencia como f para facilitar a escrita da consulta
    INNER JOIN dim_produto p -- realiza um inner join com a tabela dim_produto, referenciada como p, para combinar os dados das duas tabelas com base na chave produto_sk
        ON f.produto_sk = p.produto_sk -- especifica a condição de junção, onde a coluna produto_sk da tabela fato_vendas deve ser igual à coluna produto_sk da tabela dim_produto
    ORDER BY f.valor_total DESC -- ordena os resultados pelo valor_total em ordem decrescente
    LIMIT 10
""").df()

## Exercício 15 🟡 Médio — Ordenação para ranking

Liste as 20 vendas de maior `valor_total`.  
Mostre `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total`.

In [ ]:
con.sql("""
    SELECT
        venda_sk, 
        order_id, 
        valor_produto, 
        valor_frete, 
        valor_total
    FROM fato_vendas
    ORDER BY valor_total DESC
    LIMIT 20
""").df()

### JOIN

Em um Star Schema, normalmente a análise começa na tabela fato e depois se conecta às dimensões.

Exemplo de raciocínio:

> Quero faturamento por estado.  
> O faturamento está na `fato_vendas`.  
> O estado está na `dim_cliente`.  
> Então preciso fazer JOIN entre fato e dimensão cliente.

Tipos importantes:

- `INNER JOIN`: retorna apenas registros que possuem correspondência nos dois lados.
- `LEFT JOIN`: mantém todos os registros da tabela da esquerda, mesmo sem correspondência na direita.

### Exemplo — fato + dimensão cliente

In [ ]:
con.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    c.customer_state
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
LIMIT 10
""").df()

### Exemplo — fato + dimensão produto + dimensão tempo

In [ ]:
con.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    p.product_category_name_english AS categoria,
    t.ano_mes,
    f.valor_total
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
LIMIT 20
""").df()

#### Exemplo 2 (slide 11 - semana 9)

In [ ]:
con.sql("""
SELECT
    v.venda_sk,
    v.order_id,
    v.cliente_sk,
    c.customer_city,
    c.customer_state,
    v.valor_total
 FROM fato_vendas v
 LEFT JOIN dim_cliente c
    ON v.cliente_sk = c.cliente_sk
 ORDER BY v.valor_total DESC
 LIMIT 20;
 """).df()

#### AULA 3 - AULA PRÁTICA

## Exercício 16 🟡 Médio — Faturamento por estado

In [ ]:
con.sql("""
    SELECT
        f.venda_sk,
        f.order_id,
        f.valor_total,
        c.customer_state
    FROM fato_vendas f 
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    WHERE c.customer_state = 'SP'
    LIMIT 10
""").df()

## Exercício 17 🟡 Médio — Top 10 categorias

In [ ]:
con.sql("""
    SELECT 
        p.product_category_name_english AS categoria,
        SUM(f.valor_total) AS faturamento_total
    FROM fato_vendas f
    INNER JOIN dim_produto p 
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english
    ORDER BY faturamento_total DESC
    LIMIT 10
""").df()

## Exercício 18 🔴 Difícil — Ticket médio por estado

In [ ]:
con.sql("""
    SELECT 
        c.customer_state AS estado,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_cliente c 
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
    ORDER BY ticket_medio DESC
""").df()

## Exercício 19 🔴 Difícil — Evolução mensal de faturamento

In [ ]:
con.sql("""
    SELECT 
        t.ano_mes AS mes_ano,
        SUM(f.valor_total) AS faturamento_no_mes
    FROM fato_vendas f
    INNER JOIN dim_tempo t 
        ON f.tempo_sk = t.tempo_sk
    GROUP BY t.ano_mes
    ORDER BY t.ano_mes ASC
""").df()

In [ ]:
faturamento_mensal = con.sql("""
SELECT t.ano_mes, SUM(f.valor_total) AS faturamento_total
FROM fato_vendas f
JOIN dim_tempo t ON f.tempo_sk = t.tempo_sk
GROUP BY t.ano_mes
ORDER BY t.ano_mes
""").df()
faturamento_mensal.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(faturamento_mensal['ano_mes'], faturamento_mensal['faturamento_total'])
plt.xticks(rotation=90)
plt.title('Evolução Mensal do Faturamento')
plt.xlabel('Ano-Mês')
plt.ylabel('Faturamento')
plt.show()